In [24]:
using JLD2
using Printf
using StaticArrays
using Statistics
using Test
using WaveGreen2D.Chebyshev: ChebyshevSeries, TransformedChebyshevSeries, gradient, hessian, contains

In [2]:
@load "../src/chebyshev_series.jld2" L₁_series L₂_series;

In [3]:
function u(x::SVector{3,Float64})
    A, B, H = x

    return SA[A, B, log(H)]
end


function ∇u(x::SVector{3,Float64})
    _, _, H = x

    grad_u = one(MMatrix{3, 3, Float64})
    grad_u[3, 3] = 1/H
    
    return SMatrix(grad_u)
end


function Hu(x::SVector{3,Float64})
    _, _, H = x

    hess_u = zero(MArray{Tuple{3, 3, 3}, Float64, 3})
    hess_u[3, 3, 3] = -1/H^2

    return SArray(hess_u)
end;

In [4]:
C₁1 = L₁_series[1]
C₁2 = TransformedChebyshevSeries(L₁_series[2], u, ∇u, Hu)
C₁3 = TransformedChebyshevSeries(L₁_series[3], u, ∇u, Hu)

C₂1 = TransformedChebyshevSeries(L₂_series[1], u, ∇u, Hu)
C₂2 = TransformedChebyshevSeries(L₂_series[2], u, ∇u, Hu)
C₂3 = TransformedChebyshevSeries(L₂_series[3], u, ∇u, Hu)
C₂4 = TransformedChebyshevSeries(L₂_series[4], u, ∇u, Hu)

function C₁(x::SVector{3, Float64})
    if contains(C₁1, x)
        return C₁1(x)
    elseif contains(C₁2, x)
        return C₁2(x)
    elseif contains(C₁3, x)
        return C₁3(x)
    else
        throw(DomainError(x))
    end
end


function C₁_gradient(x::SVector{3, Float64})
    if contains(C₁1, x)
        return gradient(C₁1, x)
    elseif contains(C₁2, x)
        return gradient(C₁2, x)
    elseif contains(C₁3, x)
        return gradient(C₁3, x)
    else
        throw(DomainError(x))
    end
end


function C₁_hessian(x::SVector{3, Float64})
    if contains(C₁1, x)
        return hessian(C₁1, x)
    elseif contains(C₁2, x)
        return hessian(C₁2, x)
    elseif contains(C₁3, x)
        return hessian(C₁3, x)
    else
        throw(DomainError(x))
    end
end


function C₂(x::SVector{3, Float64})
    if contains(C₂1, x)
        return C₂1(x)
    elseif contains(C₂2, x)
        return C₂2(x)
    elseif contains(C₂3, x)
        return C₂3(x)
    elseif contains(C₂4, x)
        return C₂4(x)
    else
        throw(DomainError(x))
    end
end


function C₂_gradient(x::SVector{3, Float64})
    if contains(C₂1, x)
        return gradient(C₂1, x)
    elseif contains(C₂2, x)
        return gradient(C₂2, x)
    elseif contains(C₂3, x)
        return gradient(C₂3, x)
    elseif contains(C₂4, x)
        return gradient(C₂4, x)
    else
        throw(DomainError(x))
    end
end


function C₂_hessian(x::SVector{3, Float64})
    if contains(C₂1, x)
        return hessian(C₂1, x)
    elseif contains(C₂2, x)
        return hessian(C₂2, x)
    elseif contains(C₂3, x)
        return hessian(C₂3, x)
    elseif contains(C₂4, x)
        return hessian(C₂4, x)
    else
        throw(DomainError(x))
    end
end;

In [5]:
@load "../test/data/l1_test_data.jld2" x1 L1 ∇L1 HL1;

In [6]:
npoints = length(x1)

C1 = Vector{Float64}(undef, npoints)
C1g = Vector{Float64}(undef, npoints)
C1h = Vector{Float64}(undef, npoints)

∇C1 = Matrix{Float64}(undef, npoints, 3)
∇C1h = Matrix{Float64}(undef, npoints, 3)

HC1 = Array{Float64, 3}(undef, npoints, 3, 3);

In [7]:
for i in 1:npoints
    y₁ = C₁(x1[i])
    y₂, ∇y₂ = C₁_gradient(x1[i])
    y₃, ∇y₃, Hy₃ = C₁_hessian(x1[i])

    C1[i] = y₁
    C1g[i] = y₂
    C1h[i] = y₃
    ∇C1[i, :] .= ∇y₂
    ∇C1h[i, :] .= ∇y₃
    HC1[i, :, :] .= Hy₃
end

In [20]:
err_L = @. abs(L1 - C1)
err_∇L = @. abs(∇L1 - ∇C1)
err_HL = @. abs(HL1 - HC1)

mean_err_L = mean(err_L)
mean_err_∇L = dropdims(mean(err_∇L; dims=1); dims=1)
mean_err_HL = dropdims(mean(err_HL; dims=1); dims=1)

max_err_L = maximum(err_L)
max_err_∇L = dropdims(maximum(err_∇L; dims=1); dims=1)
max_err_HL = dropdims(maximum(err_HL; dims=1); dims=1)

std_err_L = std(err_L)
std_err_∇L = dropdims(std(err_∇L; dims=1); dims=1)
std_err_HL = dropdims(std(err_HL; dims=1); dims=1)

lim_err_L = mean_err_L + 3 * std_err_L
lim_err_∇L = reshape(mean_err_∇L + 3 * std_err_∇L, 1, 3)
lim_err_HL = reshape(mean_err_HL + 3 * std_err_HL, 1, 3, 3)

pct_err_L = sum(err_L .< lim_err_L) * 100 / npoints
pct_err_∇L = dropdims(count(err_∇L .< lim_err_∇L; dims=1); dims=1) * 100 / npoints
pct_err_HL = dropdims(sum(err_HL .< lim_err_HL; dims=1); dims=1) * 100 / npoints;

In [21]:
max_err_∇L

3-element Vector{Float64}:
 1.8494559182735681e-13
 1.978417429882029e-13
 3.480549182199866e-13

In [12]:
result = """
L₁c mean absolute error = $(@sprintf("%.0e", mean_err_L))

∇L₁c mean absolute error = $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_∇L...))

HL₁c mean absolute error = $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[1, :]...))
                           $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[2, :]...))
                           $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[3, :]...))

For L₁c, ∇L₁c and HL₁c, the error value for each point is compared with a corresponding reference
value. The reference value is the mean absolute error plus three times the standard deviation. The
following results display the percentage of points for which the error is less than the reference.

Percentage of L₁c error values lower than reference = $(@sprintf("%.1f%%", pct_err_L))

Percentage of ∇L₁c error values lower than reference = $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_∇L...))

Percentage of HL₁c error values lower than reference = $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[1, :]...))
                                                       $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[2, :]...))
                                                       $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[3, :]...))
"""

println(result)

L₁c mean absolute error = 6e-16

∇L₁c mean absolute error = [2e-14  1e-14  7e-15]

HL₁c mean absolute error = [2e-12  6e-13  3e-13]
                           [6e-13  9e-13  2e-13]
                           [3e-13  2e-13  3e-13]

For L₁c, ∇L₁c and HL₁c, the error value for each point is compared with a corresponding reference
value. The reference value is the mean absolute error plus three times the standard deviation. The
following results display the percentage of points for which the error is less than the reference.

Percentage of L₁c error values lower than reference = 99.2%

Percentage of ∇L₁c error values lower than reference = [97.8%  98.2%  99.2%]

Percentage of HL₁c error values lower than reference = [97.9%  98.5%  98.5%]
                                                       [98.5%  98.7%  98.5%]
                                                       [98.5%  98.5%  99.2%]



In [79]:
tol_L = 10^2 * eps()
tol_∇L = 10^4 * eps()
tol_HL = 10^6 * eps()

2.220446049250313e-10

In [83]:
@testset "L₁ Chebyshev" begin
    @load "../test/data/l1_test_data.jld2" x1 L1 ∇L1 HL1

    npoints = length(x1)
    
    C1 = Vector{Float64}(undef, npoints)
    C1g = Vector{Float64}(undef, npoints)
    C1h = Vector{Float64}(undef, npoints)
    
    ∇C1 = Matrix{Float64}(undef, npoints, 3)
    ∇C1h = Matrix{Float64}(undef, npoints, 3)
    
    HC1 = Array{Float64, 3}(undef, npoints, 3, 3)

    for i in 1:npoints
        y₁ = C₁(x1[i])
        y₂, ∇y₂ = C₁_gradient(x1[i])
        y₃, ∇y₃, Hy₃ = C₁_hessian(x1[i])
    
        C1[i] = y₁
        C1g[i] = y₂
        C1h[i] = y₃
        ∇C1[i, :] .= ∇y₂
        ∇C1h[i, :] .= ∇y₃
        HC1[i, :, :] .= Hy₃
    end

    # Test if ChebyshevSeries, gradient and hessian return the same value of L₁
    @test C1 == C1g
    @test C1 == C1h

    # Test if gradient and hessian return the same value of ∇L₁
    @test ∇C1 == ∇C1h

    @test all(isapprox.(L1, C1; rtol=tol_L, atol=tol_L))
    @test all(isapprox.(∇L1, ∇C1; rtol=tol_∇L, atol=tol_∇L))
    @test all(isapprox.(HL1, HC1; rtol=tol_HL, atol=tol_HL))
end

Test Summary: | Pass  Total  Time
L₁ Chebyshev  |    6      6  0.1s


Test.DefaultTestSet("L₁ Chebyshev", Any[], 6, false, false, true, 1.782325484914215e9, 1.782325484987663e9, false, "In[83]", Random.Xoshiro(0x3c9667e9e49ca270, 0x406dcf0562f5426e, 0x0168e9b257dd7f35, 0xbb4b177f61386665, 0x23fa09f16d5ab286))

In [82]:
@testset "L₂ Chebyshev" begin
    @load "../test/data/l2_test_data.jld2" x2 L2 ∇L2 HL2

    npoints = length(x2)
    
    C2 = Vector{Float64}(undef, npoints)
    C2g = Vector{Float64}(undef, npoints)
    C2h = Vector{Float64}(undef, npoints)
    
    ∇C2 = Matrix{Float64}(undef, npoints, 3)
    ∇C2h = Matrix{Float64}(undef, npoints, 3)
    
    HC2 = Array{Float64, 3}(undef, npoints, 3, 3)

    for i in 1:npoints
        y₁ = C₂(x2[i])
        y₂, ∇y₂ = C₂_gradient(x2[i])
        y₃, ∇y₃, Hy₃ = C₂_hessian(x2[i])
    
        C2[i] = y₁
        C2g[i] = y₂
        C2h[i] = y₃
        ∇C2[i, :] .= ∇y₂
        ∇C2h[i, :] .= ∇y₃
        HC2[i, :, :] .= Hy₃
    end

    # Test if ChebyshevSeries, gradient and hessian return the same value of L₁
    @test C2 == C2g
    @test C2 == C2h

    # Test if gradient and hessian return the same value of ∇L₁
    @test ∇C2 == ∇C2h

    @test all(isapprox.(L2, C2; rtol=tol_L, atol=tol_L))
    @test all(isapprox.(∇L2, ∇C2; rtol=tol_∇L, atol=tol_∇L))
    @test all(isapprox.(HL2, HC2; rtol=tol_HL, atol=tol_HL))
end

Test Summary: | Pass  Total  Time
L₂ Chebyshev  |    6      6  0.1s


Test.DefaultTestSet("L₂ Chebyshev", Any[], 6, false, false, true, 1.782325448243267e9, 1.782325448328262e9, false, "In[82]", Random.Xoshiro(0x3c9667e9e49ca270, 0x406dcf0562f5426e, 0x0168e9b257dd7f35, 0xbb4b177f61386665, 0x23fa09f16d5ab286))

In [39]:
@load "../test/data/l2_test_data.jld2" x2 L2 ∇L2 HL2

npoints = length(x2)

C2 = Vector{Float64}(undef, npoints)
C2g = Vector{Float64}(undef, npoints)
C2h = Vector{Float64}(undef, npoints)

∇C2 = Matrix{Float64}(undef, npoints, 3)
∇C2h = Matrix{Float64}(undef, npoints, 3)

HC2 = Array{Float64, 3}(undef, npoints, 3, 3)

for i in 1:npoints
    y₁ = C₂(x2[i])
    y₂, ∇y₂ = C₂_gradient(x2[i])
    y₃, ∇y₃, Hy₃ = C₂_hessian(x2[i])

    C2[i] = y₁
    C2g[i] = y₂
    C2h[i] = y₃
    ∇C2[i, :] .= ∇y₂
    ∇C2h[i, :] .= ∇y₃
    HC2[i, :, :] .= Hy₃
end

# Statistical data
err_L = @. abs(L2 - C2)
err_∇L = @. abs(∇L2 - ∇C2)
err_HL = @. abs(HL2 - HC2)

mean_err_L = mean(err_L)
mean_err_∇L = dropdims(mean(err_∇L; dims=1); dims=1)
mean_err_HL = dropdims(mean(err_HL; dims=1); dims=1)

max_err_L = maximum(err_L)
max_err_∇L = dropdims(maximum(err_∇L; dims=1); dims=1)
max_err_HL = dropdims(maximum(err_HL; dims=1); dims=1);

In [47]:
abs_err_L = @. abs(L2 - C2)
abs_err_∇L = @. abs(∇L2 - ∇C2)
abs_err_HL = @. abs(HL2 - HC2)

rel_err_L = @. abs(L2 - C2) / L2
rel_err_∇L = @. abs(∇L2 - ∇C2) / ∇L2
rel_err_HL = @. abs(HL2 - HC2) / HL2;

In [74]:
test = isapprox.(L2, C2; rtol=tol_L, atol=tol_L)
findall(.!test)

1-element Vector{Int64}:
 390

In [75]:
x2[390]

3-element SVector{3, Float64} with indices SOneTo(3):
 0.4909137049586571
 1.4115391338743177
 0.3318598021875803

In [76]:
L2[390] - C2[390]

-3.1086244689504383e-15

In [77]:
tol_L

2.220446049250313e-15

In [72]:
test = isapprox.(∇L2, ∇C2; rtol=tol_∇L, atol=tol_∇L)
findall(.!test)

CartesianIndex{2}[]

In [73]:
all(test)

true

In [69]:
∇L2 - ∇C2

1000×3 Matrix{Float64}:
 -1.67644e-14  -1.24345e-14  -6.32827e-15
  2.32453e-16   2.66454e-15  -5.12437e-15
 -5.80449e-14   3.85803e-15  -4.88498e-15
 -2.498e-16     8.65974e-15   1.80411e-15
  2.87964e-16  -1.03251e-14   8.36137e-16
 -2.58127e-15   7.77156e-16  -5.28744e-15
  3.76088e-15  -3.33067e-15  -2.498e-16
  3.07046e-16   1.9984e-15   -7.54952e-15
  1.87849e-15   4.44089e-16  -5.89459e-15
 -1.6355e-14   -1.34337e-14   7.07767e-16
 -2.06779e-15  -4.44089e-16   4.20497e-15
 -9.8207e-16    5.55112e-16  -1.62093e-14
 -1.90646e-15  -1.55431e-15  -2.98372e-16
  ⋮                          
 -1.70081e-14  -3.88578e-16  -3.4639e-14
 -8.02601e-15   1.11022e-15  -4.12517e-15
  3.11036e-15   3.88578e-15  -4.16334e-15
 -7.98667e-15   1.26565e-14  -6.82787e-15
 -7.46625e-15  -5.9952e-15    1.02141e-14
 -3.67761e-16  -5.21805e-15  -4.85723e-16
 -1.73472e-16  -2.55351e-15   2.13718e-15
  6.1548e-15    3.55271e-15   4.02456e-16
 -3.47639e-15  -7.10543e-15  -9.4369e-15
 -1.94289e-15   5.76275e-1

In [52]:
using LinearAlgebra

In [53]:
norm(L2)

25.961584077605533

In [54]:
norm(∇L2)

193.63337223238304

In [55]:
norm(HL2)

6681.811526895514

In [57]:
maximum(abs.(∇L2))

115.26649327373781

In [56]:
maximum(abs.(HL2))

6084.4521487317415